In [ ]:
from dotenv import load_dotenv


load_dotenv("../resume-parser-core/.env")

In [56]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

GROQ_SYSTEM_MESSAGE = """Your name is Kaross, and you are a career consultant. Your task is to answer users' career-related questions such as resume tips, job and market insights, etc. You should be as concise and constructive as possible. For questions you don't know the answer to, you can try to use the search tool to find the answer."""

GEMINI_SYSTEM_MESSAGE = """You are a helpful AI assistant. You are here to answer questions raised by the users. If you don't know the answer, just say you don't know. Try to make your answers as concise as possible."""

groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=3,
    streaming=True,
)

gemini_llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash-exp",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=3,
)

# Graph State

In [57]:
from typing import TypedDict, Annotated, Sequence
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage


class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Tools

In [60]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool


@tool
async def search_tool(query: str) -> str:
    """Search for information using Google Search. Pass in a complete question to get a clear answer."""

    # The description of the tool is a lie, it's actually to ask gemini for help. If I have more money for API keys I would use another tool.
    prompt = ChatPromptTemplate(
        [
            ("system", GEMINI_SYSTEM_MESSAGE),
            ("user", "{query}"),
        ]
    )
    chain = prompt | gemini_llm
    result = await chain.ainvoke({"query": query})
    return {"result": result.content}


tools = [search_tool]
groq_agent = groq_llm.bind_tools(tools)

# Graph Nodes

In [52]:
import json
from langchain_core.messages import SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig

tools_by_name = {tool.name: tool for tool in tools}


async def tool_node(state: State):
    """Call the tools with the arguments provided in the last message"""

    outputs = []
    for tool_call in state["messages"][-1].tool_calls:
        tool_result = await tools_by_name[tool_call["name"]].ainvoke(tool_call["args"])
        outputs.append(
            ToolMessage(
                content=json.dumps(tool_result),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )
    return {"messages": outputs}


async def call_model(
    state: State,
    config: RunnableConfig,
):
    """Call the model with the system prompt and the conversation history"""

    system_prompt = SystemMessage(GROQ_SYSTEM_MESSAGE)
    response = await groq_agent.ainvoke([system_prompt] + state["messages"], config)
    return {"messages": [response]}


def should_continue(state: State):
    """Conditional edge to check if agent needs to use tools"""

    messages = state["messages"]
    last_message = messages[-1]

    # If there is no function call, then we finish
    if not last_message.tool_calls:
        return "end"

    # Otherwise if there is, we continue
    else:
        return "continue"

In [53]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


memory = MemorySaver()
graph = StateGraph(state_schema=State)

graph.add_node("tool_node", tool_node)
graph.add_node("call_model", call_model)

graph.add_edge(START, "call_model")
graph.add_conditional_edges(
    "call_model",
    should_continue,
    {"continue": "tool_node", "end": END},
)
graph.add_edge("tool_node", "call_model")

chatflow = graph.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

display(Image(chatflow.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import AIMessageChunk, HumanMessage

while True:
    user_input = input("User: ")
    if user_input == "quit":
        break

    config = {"configurable": {"thread_id": "1"}}
    first = True

    async for msg, metadata in chatflow.astream(
        {"messages": HumanMessage(content=user_input)},
        stream_mode="messages",
        config=config,
    ):
        if (
            msg.content
            and isinstance(msg, AIMessageChunk)
            and metadata["langgraph_node"] == "call_model"
        ):
            print(msg.content, end="", flush=True)

        # Print out tool messages if needed
        # if isinstance(msg, AIMessageChunk):
        #     if first:
        #         gathered = msg
        #         first = False
        #     else:
        #         gathered = gathered + msg

        #     if msg.tool_call_chunks:
        #         print(gathered.tool_calls)